In [1]:
import sys
from pathlib import Path
import cv2
import numpy as np

REPO_ROOT        = Path("..").resolve()
ORIG_FRAMES_DIR  = REPO_ROOT / "video_data" / "decomposed_frames"
EDITED_FRAMES_DIR = REPO_ROOT / "video_data" / "edited_video_frames"
JND_MAPS_DIR     = REPO_ROOT / "video_data" / "jnd_maps"

In [6]:
total_violations = 0
total_pixels = 0
violation_frames = []

orig_frame_paths = sorted(ORIG_FRAMES_DIR.glob("frame_*.png"))
print(f"Found {len(orig_frame_paths)} frames")

for orig_path in orig_frame_paths:
    num = orig_path.stem.split("_", 1)[1]   # "frame_0001" -> "0001"

    edited_path = EDITED_FRAMES_DIR / f"frame_{num}.png"
    jnd_path    = JND_MAPS_DIR      / f"jnd_{num}.png"

    # Load original frame as RGB
    orig_bgr = cv2.imread(str(orig_path))
    orig_rgb = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB).astype(np.int32)
    R, G, B  = orig_rgb[:, :, 0], orig_rgb[:, :, 1], orig_rgb[:, :, 2]

    # Custom grayscale formula: floor((299*R + 587*G + 114*B) / 1000)
    custom_gray = ((299 * R + 587 * G + 114 * B) // 1000).astype(np.uint8)

    # Load edited frame (already grayscale; read as grayscale to be safe)
    edited_gray = cv2.imread(str(edited_path), cv2.IMREAD_GRAYSCALE).astype(np.int32)

    # Load JND map (8-bit grayscale, values are floor'd thresholds)
    jnd_map = cv2.imread(str(jnd_path), cv2.IMREAD_GRAYSCALE).astype(np.int32)

    diff = np.abs(custom_gray.astype(np.int32) - edited_gray)
    violations = int(np.sum(diff > 5*jnd_map))

    total_violations += violations
    total_pixels     += custom_gray.size
    if violations > 0:
        violation_frames.append((num, violations))

print(f"\nTotal pixels checked : {total_pixels:,}")
print(f"Total JND violations : {total_violations:,}")
print(f"Frames with violations: {len(violation_frames)} / {len(orig_frame_paths)}")
if violation_frames:
    print("\nPer-frame violation counts:")
    for num, count in violation_frames:
        print(f"  frame {num}: {count:,} violations")

Found 300 frames

Total pixels checked : 276,480,000
Total JND violations : 44,049
Frames with violations: 300 / 300

Per-frame violation counts:
  frame 0001: 9 violations
  frame 0002: 20 violations
  frame 0003: 59 violations
  frame 0004: 125 violations
  frame 0005: 24 violations
  frame 0006: 118 violations
  frame 0007: 166 violations
  frame 0008: 93 violations
  frame 0009: 18 violations
  frame 0010: 114 violations
  frame 0011: 120 violations
  frame 0012: 220 violations
  frame 0013: 27 violations
  frame 0014: 155 violations
  frame 0015: 159 violations
  frame 0016: 145 violations
  frame 0017: 29 violations
  frame 0018: 225 violations
  frame 0019: 126 violations
  frame 0020: 108 violations
  frame 0021: 26 violations
  frame 0022: 105 violations
  frame 0023: 110 violations
  frame 0024: 265 violations
  frame 0025: 21 violations
  frame 0026: 143 violations
  frame 0027: 177 violations
  frame 0028: 132 violations
  frame 0029: 22 violations
  frame 0030: 212 violati

In [9]:
import time
import random

# jnd_diff.py lives in the same directory as this notebook
sys.path.insert(0, str(Path(__file__).parent if "__file__" in dir() else Path(".").resolve()))
from jnd_diff import compute_jnd_map, load_gray

frame_paths = sorted(EDITED_FRAMES_DIR.glob("frame_*.png"))
frame_path  = random.choice(frame_paths)
print(f"Benchmarking on: {frame_path.name}")

img = load_gray(str(frame_path))

N = 5
times = []
for _ in range(N):
    t0 = time.perf_counter()
    compute_jnd_map(img)
    times.append(time.perf_counter() - t0)

print(f"  Runs         : {N}")
print(f"  Min          : {min(times)*1000:.1f} ms")
print(f"  Max          : {max(times)*1000:.1f} ms")
print(f"  Mean         : {sum(times)/len(times)*1000:.1f} ms")
print(f"  Image size   : {img.shape[1]}x{img.shape[0]} (WxH)")

Benchmarking on: frame_0239.png
  Runs         : 5
  Min          : 473.7 ms
  Max          : 477.8 ms
  Mean         : 475.9 ms
  Image size   : 1280x720 (WxH)
